# IMPORTAZIONE DELLE LIBRERIE


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import os
import requests


import pandas as pd
from pathlib import Path

In [ ]:
os.getcwd()

In [ ]:
if not os.path.exists("../Data/Fetching_data"):
    os.mkdir("Fetching_data")

# SCRAPING DA SITUAS

In [ ]:
url_situas = "https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31"

resp_situas = requests.get (url_situas) 

print("Risposta alla richiesta:", resp_situas.status_code)

In [ ]:
# =============================
# COSTANTI
# =============================

# URL del report SITUAS
base_url = "https://situas.istat.it/web/#/territorio/body"

# ID del report
report_id = 74

# Intervallo di anni da scaricare
anno_inizio = 2001
anno_fine = 2024

# timeout massimo delle attese 
timeout = 120

# Cartella (relativa al progetto) in cui salvare i CSV
download_dir = Path("../Data/Fetching_data") 
download_dir.mkdir(parents=True, exist_ok=True)

# ========================================
# CONFIGURAZIONE DI CHROME
# =========================================

options = webdriver.ChromeOptions()

prefs = {# Cartella di download
        "download.default_directory": str(download_dir.resolve()),

        # Disabilita la richiesta di conferma del download
        "download.prompt_for_download": False,

        # Utilizza sempre la cartella specificata
        "download.directory_upgrade": True}

options.add_experimental_option("prefs", prefs)

# Avvio del browser
driver = webdriver.Chrome(options=options)

# Oggetto utilizzato per le attese esplicite
wait = WebDriverWait(driver, timeout)

# ==============================
# SELETTORI DA PREMERE
# ==============================

export_button = (By.ID, "dati-report-export-btn")

csv_button = (By.CSS_SELECTOR, 'button[title="Scarica dati in formato CSV"]')

# ==============================
# FUNZIONI
# ===============================

def apri_report(anno):
    """
    Apre il report relativo all'anno richiesto.
    """

    url = (f"{base_url}"
        f"?id={report_id}"
        f"&dateFrom={anno}-12-31")

    driver.get(url)

    # Attende il caricamento completo della pagina
    wait.until(
        lambda d: d.execute_script("return document.readyState") == "complete")


def clicca_esporta():
    """
    Apre il menu di esportazione.
    """

    export = wait.until(EC.element_to_be_clickable(export_button))

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",export)

    driver.execute_script(
        "arguments[0].click();",export)


def clicca_csv():
    """
    Clicca sul pulsante CSV ed avvia il download.
    """

    csv = wait.until(EC.element_to_be_clickable(csv_button))

    driver.execute_script("arguments[0].click();",csv)

    # Attesa affinché Chrome inizi il download
    time.sleep(3)


def attendi_download(file_prima):
    """
    Attende che venga scaricato un nuovo file CSV.

    Parameters
    ----------
    file_prima : set
        Insieme dei file presenti prima del download.

    Returns
    -------
    Path
        Percorso del nuovo CSV scaricato.
    """

    # Attende che inizi il download
    wait.until(lambda d:
               len(list(download_dir.glob("*.crdownload"))) > 0
                or
                len(set(download_dir.glob("*.csv")) - file_prima) > 0)

    # Attende la fine del download
    wait.until(lambda d:
            len(list(download_dir.glob("*.crdownload"))) == 0)

    # Individua il nuovo CSV scaricato
    nuovi_file = list(set(download_dir.glob("*.csv")) - file_prima)

    return nuovi_file[0]


def rinomina_csv(file_csv, anno):
    """
    Rinomina il CSV appena scaricato.
    """

    nuovo_nome = download_dir / f"situas_{anno}.csv"

    os.replace(file_csv, nuovo_nome)



In [ ]:
# ==============================
# DOWNLOAD DEI REPORT
# ==============================

for anno in range(anno_inizio, anno_fine + 1):

    output_file = download_dir / f"situas_{anno}.csv"

    if output_file.exists():
        print(f"{output_file.name} esistente - download saltato")
        continue

    print(f"\n========== {anno} ==========")

    # Memorizza i file presenti prima del download
    file_prima = set(download_dir.glob("*.csv"))

    print("Apro il report...")
    apri_report(anno)

    print("Apro il menu Esporta...")
    clicca_esporta()

    print("Avvio il download del CSV...")
    clicca_csv()

    print("Attendo il completamento del download...")
    file_csv = attendi_download(file_prima)

    print("Rinomino il file...")
    rinomina_csv(file_csv, anno)

    print(f"-Report {anno} scaricato correttamente-")

    # Piccola pausa prima dell'anno successivo
    time.sleep(2)

# ================================
# CHIUSURA DEL BROWSER
# ================================
driver.quit()

print("\nDownload completato con successo!")

In [ ]:
# =======================================
# LETTURA E UNIONE DEI CSV SITUAS
# =======================================

# Cartella contenente i CSV scaricati
download_dir = Path("../Data/Fetching_data")

# Elenco dei file CSV
file_csv = sorted(download_dir.glob("*.csv"))

# Lista dei DataFrame
lista_df = []

# Lettura dei file
for file in file_csv:

    print(f"Lettura di {file.name}")

    # Lettura del CSV (aggiungi sep=';' se necessario)
    df = pd.read_csv(file, sep=";")

    # Estrae l'anno dal nome del file
    # Es.: situas_2001.csv -> 2001
    anno = int(file.stem.split("_")[1])

    df["TIME_PERIOD"]=anno #questo mi sarà utile per la merge con istat, perchè così dovrei avere comune univoco per annno
                                #univoco, e quindi queste due colonne verranno joinate con il df istat

    # Aggiunge il DataFrame alla lista
    lista_df.append(df)

# Unione di tutti i DataFrame
df_situas = pd.concat(lista_df,ignore_index=True)

# Controllo
print(df_situas.shape)
display(df_situas.head(20))


In [ ]:
df_situas.nunique()

In [ ]:
def check_colonne(df):

    for colonna in df.columns:

        print("=" * 70)
        print(f"COLONNA: {colonna}")
        print("=" * 70)

        print(f"Tipo: {df[colonna].dtype}")
        print(f"Valori mancanti: {df[colonna].isna().sum()}")
        print(f"Valori unici: {df[colonna].nunique()}")

        print("\nValori più frequenti:")

        print(
            df[colonna]
            .value_counts(dropna=False)
            .head(10)
        )

        print("\n")

In [ ]:
check_colonne(df_situas)

In [ ]:
#dalle informazioni di df_situas ho visto che ci sono 24 comuni senza nome, li cerco
df_situas[df_situas["Comune"].isna()]

In [ ]:
#Sostituisco i valori letti come NaN con il nome vero del comune "None" solo che
#devo aggiungere un "." sennò lo leggerebbe come valore nullo
df_situas["Comune"] = df_situas["Comune"].fillna("None.")

In [ ]:
df_situas["Comune"].isna().sum()

In [ ]:
#se non esiste la cartella df raw me la faccio creare
if not os.path.exists("../Data/Df_raw"):
    os.mkdir("Df_raw")

In [ ]:
#e ci salvo dentro il mio df situas grezzo

df_situas.to_csv("../Data/Df_raw/df_situas_raw.csv", index=False)

# RICHIAMO API per ISTAT

Tramite richiesta API scarico il dataset di istat 

In [ ]:
url_istat = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

http_header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

resp_istat = requests.get(url_istat, headers= http_header )

print("Risposta alla richiesta:", resp_istat.status_code)

In [ ]:
#Impongo condizione per cui se non esiste il file incidenti_istat venga creato e si scriva la risposta all' API (il dataset)
if not os.path.exists("../Data/Df_raw/df_istat_raw.csv"):
    istat_incidents = open("../Data/Df_raw/df_istat_raw.csv", "w", encoding="utf-8") #utf-8 serve per avere i caratteri corretti
    istat_incidents.write(resp_istat.text)
    istat_incidents.close()

In [ ]:
df_istat_raw= pd.read_csv("../Data/Df_raw/df_istat_raw.csv")
df_istat_raw

In [ ]:
df_istat_raw.info()

In [ ]:
check_colonne(df_istat_raw)